# 🏗️ Day 02a: Keras Sequential — Build, Compile, Fit

---

## 🎯 What You'll Master Today
- `Sequential` model — layer stacking
- `compile` — optimizer, loss, metrics
- `fit` — training loop in one call
- `evaluate` & `predict`
- Callbacks: EarlyStopping, ModelCheckpoint

**Goal:** Build, train, and save a model from memory.

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  Keras Workflow                                                  ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  1. model = Sequential([layers...])                             ║
║  2. model.compile(optimizer, loss, metrics)                     ║
║  3. model.fit(X, y, epochs, validation_split, callbacks)        ║
║  4. model.evaluate(X_test, y_test)                              ║
║  5. model.predict(X_new)                                        ║
║                                                                   ║
║  Compile cheatsheet:                                             ║
║    Binary class → loss='binary_crossentropy', sigmoid output    ║
║    Multi class  → loss='sparse_categorical_crossentropy', softmax║
║    Regression   → loss='mse', linear output                     ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

# ============================================================
# 1. Binary Classification — Sequential Model
# ============================================================

# Synthetic data
np.random.seed(42)
X = np.random.randn(1000, 10).astype(np.float32)
y = (X[:, 0] + X[:, 1] > 0).astype(np.float32)  # binary labels

model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(10,)),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid'),  # binary output
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

model.summary()
print("\n💡 Dense layers = PyTorch's nn.Linear")
print("   Keras includes activation in Dense (PyTorch separates them)")

In [ ]:
# ============================================================
# 2. Training with fit() — The Core Loop
# ============================================================

history = model.fit(
    X, y,
    epochs=20,
    batch_size=32,
    validation_split=0.2,  # 20% held out as validation
    verbose=1,
)

# history.history contains per-epoch metrics
print(f"\nFinal train acc: {history.history['accuracy'][-1]:.3f}")
print(f"Final val acc:   {history.history['val_accuracy'][-1]:.3f}")
print(f"History keys:    {list(history.history.keys())}")

In [ ]:
# ============================================================
# 3. Evaluate & Predict
# ============================================================

# Test data
X_test = np.random.randn(200, 10).astype(np.float32)
y_test = (X_test[:, 0] + X_test[:, 1] > 0).astype(np.float32)

# Evaluate
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test loss: {loss:.4f}, Test accuracy: {acc:.3f}")

# Predict
preds = model.predict(X_test[:5], verbose=0)
print(f"\nPredictions (probabilities): {preds.flatten()}")
print(f"Predicted classes: {(preds.flatten() > 0.5).astype(int)}")
print(f"True labels:       {y_test[:5].astype(int)}")

In [ ]:
# ============================================================
# 4. Multi-class Classification
# ============================================================

# Synthetic multi-class data
X_multi = np.random.randn(1000, 20).astype(np.float32)
y_multi = np.random.randint(0, 5, 1000)  # 5 classes, integer labels

model_multi = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(20,)),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dense(5, activation='softmax'),  # 5 classes
])

model_multi.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',  # integer labels
    metrics=['accuracy'],
)

model_multi.fit(X_multi, y_multi, epochs=10, batch_size=32,
                validation_split=0.2, verbose=0)
print("Multi-class model trained!")
print(f"Output shape: {model_multi.predict(X_multi[:1], verbose=0).shape}")

print("""
╔════════════════════════════════════════════════════╗
║  Loss Function Decision Table                     ║
╠════════════════════════════════════════════════════╣
║  Task            │ Output  │ Loss                 ║
║  ─────────────── │ ─────── │ ──────────────────── ║
║  Binary class    │ sigmoid │ binary_crossentropy  ║
║  Multi (int y)   │ softmax │ sparse_categorical_CE║
║  Multi (one-hot) │ softmax │ categorical_CE       ║
║  Regression      │ linear  │ mse / mae            ║
╚════════════════════════════════════════════════════╝
""")

In [ ]:
# ============================================================
# 5. Callbacks — EarlyStopping & ModelCheckpoint
# ============================================================

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,           # stop if val_loss doesn't improve for 5 epochs
        restore_best_weights=True,  # go back to best epoch
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,           # halve LR
        patience=3,           # after 3 epochs of no improvement
    ),
]

# Retrain with callbacks
model2 = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(10,)),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid'),
])
model2.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history2 = model2.fit(
    X, y, epochs=100, batch_size=32,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=0,
)
print(f"Trained for {len(history2.history['loss'])} epochs (early stopped)")

# Save & Load
# model2.save('model.keras')     # SavedModel format
# loaded = keras.models.load_model('model.keras')
print("\n💡 EarlyStopping + restore_best_weights = automatic best model")

---

## 🗺️ Keras Sequential Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  KERAS SEQUENTIAL WORKFLOW                                       ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Build:    Sequential([Dense, BN, Dropout, ...])                ║
║  Compile:  optimizer + loss + metrics                           ║
║  Train:    model.fit(X, y, epochs, validation_split, callbacks) ║
║  Evaluate: model.evaluate(X_test, y_test)                       ║
║  Predict:  model.predict(X_new)                                 ║
║                                                                   ║
║  Key callbacks:                                                  ║
║    EarlyStopping(patience, restore_best_weights)                ║
║    ModelCheckpoint(filepath, save_best_only)                    ║
║    ReduceLROnPlateau(factor, patience)                          ║
║                                                                   ║
║  history.history → dict of metrics per epoch                    ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | When to use sparse_categorical vs categorical crossentropy? | sparse = integer labels (0,1,2...). categorical = one-hot encoded labels |
| 2 | What does compile() do? | Sets optimizer, loss, and metrics. Does NOT train. Prepares the model graph for training |
| 3 | How does EarlyStopping work? | Monitors a metric (val_loss), stops training if no improvement for `patience` epochs, optionally restores best weights |
| 4 | validation_split vs validation_data? | split = take last X% of training data. validation_data = explicit (X_val, y_val) tuple |
| 5 | How to save/load Keras models? | model.save('path.keras'), keras.models.load_model('path.keras'). Saves architecture + weights + optimizer state |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Sequential([layers]) → compile → fit → evaluate     │
## │  • Binary: sigmoid + binary_crossentropy                │
## │  • Multi: softmax + sparse_categorical_crossentropy     │
## │  • EarlyStopping + restore_best_weights = must use      │
## │  • history.history for training curves                   │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Functional API** — complex architectures